In [1]:
import gmsh
import numpy as np
from mpi4py import MPI
from dolfinx.io import XDMFFile, gmshio
from dolfinx.io.gmshio import read_from_msh

In [2]:

def draw_mesh(r1, r2, angle):
    gmsh.initialize()
    gmsh.clear()
    gmsh.model.add("test")
    gmsh.model.setCurrent("test")

    gmsh.model.occ.addPoint(250e-3,0,0,meshSize=10e-3, tag=1)
    gmsh.model.occ.addPoint(250e-3, 500e-3, 0, meshSize=10e-3, tag=2)
    gmsh.model.occ.addPoint(0,0,0,meshSize=10e-3, tag=3)
    gmsh.model.occ.addPoint(0,500e-3,0, meshSize=10e-3, tag=4)

    gmsh.model.occ.addLine(3,1,tag=1)
    gmsh.model.occ.addLine(3,4,tag=2)
    gmsh.model.occ.addLine(1,2,tag=3)
    gmsh.model.occ.addLine(2,4,tag=4)

    gmsh.model.occ.addCurveLoop([3,4,-2,1], tag=1)


    
    gmsh.model.occ.addPoint(80e-3+r1, 150e-3, 0, meshSize=4e-3, tag=20)   
    gmsh.model.occ.addEllipse(80e-3,150e-3,0, r1,r2, tag=5, angle1=0, angle2=2*np.pi,) 
    gmsh.model.occ.rotate([(0,20),(1,5)],80e-3,150e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([5], tag=2)

    gmsh.model.occ.addPoint(170e-3+r1, 150e-3, 0, meshSize=4e-3, tag=22)
    gmsh.model.occ.addEllipse(170e-3,150e-3,0, r1,r2, tag=6, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,22),(1,6)],170e-3,150e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([6], tag=3)

    gmsh.model.occ.addPoint(80e-3+r1, 250e-3, 0, meshSize=4e-3, tag=24)
    gmsh.model.occ.addEllipse(80e-3,250e-3,0, r1,r2, tag=7, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,24),(1,7)],80e-3,250e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([7], tag=4)

    
    gmsh.model.occ.addPoint(170e-3+r1, 250e-3, 0, meshSize=4e-3, tag=26)
    gmsh.model.occ.addEllipse(170e-3,250e-3,0, r1,r2, tag=8, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,26),(1,8)],170e-3,250e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([8], tag=5)

    gmsh.model.occ.addPoint(80e-3+r1, 350e-3, 0, meshSize=4e-3, tag=28)
    gmsh.model.occ.addEllipse(80e-3,350e-3,0, r1,r2, tag=9, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,28),(1,9)],80e-3,350e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([9], tag=6)

    gmsh.model.occ.addPoint(170e-3+r1, 350e-3, 0, meshSize=4e-3, tag=30)
    gmsh.model.occ.addEllipse(170e-3,350e-3,0, r1,r2, tag=10, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,30),(1,10)],170e-3,350e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([10], tag=7)


    gmsh.model.occ.addPlaneSurface([1,2,3,4,5,6,7], tag=1)


    gmsh.model.occ.synchronize()

    gmsh.model.addPhysicalGroup(1, [1,2,3,4], 1, "V=0")
    gmsh.model.addPhysicalGroup(1, [ 5,8,9], 2, "V=-1")
    gmsh.model.addPhysicalGroup(1, [ 6,7,10], 3, "V=1")
    
    
    
    gmsh.model.addPhysicalGroup(2, [1], 5, "domain")

    gmsh.model.mesh.generate(2)
    gmsh.model.mesh.optimize("Netgen")

    model_rank = 0
    mesh_comm = MPI.COMM_WORLD
    domain, ct, ft = gmshio.model_to_mesh(gmsh.model, mesh_comm, model_rank, gdim=2)
    gmsh.finalize()
    return domain, ft

In [3]:
"""
import pyvista
from dolfinx.plot import vtk_mesh

domain, ct=draw_mesh(40e-3, 10e-3, 30)
pyvista.start_xvfb()
plotter = pyvista.Plotter()
tdim = domain.topology.dim
domain.topology.create_connectivity(tdim, tdim)
grid = pyvista.UnstructuredGrid(*vtk_mesh(domain, tdim))
num_local_cells = domain.topology.index_map(tdim).size_local
grid.cell_data["Marker"] = ct.values[ct.indices < num_local_cells]
grid.set_active_scalars("Marker")
actor = plotter.add_mesh(grid, show_edges=True, color="white")
plotter.view_xy()

plotter.show()
"""

'\nimport pyvista\nfrom dolfinx.plot import vtk_mesh\n\ndomain, ct=draw_mesh(40e-3, 10e-3, 30)\npyvista.start_xvfb()\nplotter = pyvista.Plotter()\ntdim = domain.topology.dim\ndomain.topology.create_connectivity(tdim, tdim)\ngrid = pyvista.UnstructuredGrid(*vtk_mesh(domain, tdim))\nnum_local_cells = domain.topology.index_map(tdim).size_local\ngrid.cell_data["Marker"] = ct.values[ct.indices < num_local_cells]\ngrid.set_active_scalars("Marker")\nactor = plotter.add_mesh(grid, show_edges=True, color="white")\nplotter.view_xy()\n\nplotter.show()\n'

In [4]:
import os
from dolfinx import mesh, fem, plot, io, default_scalar_type
from dolfinx.fem.petsc import LinearProblem
from mpi4py import MPI
import ufl
import numpy as np
from dolfinx import geometry
from petsc4py import PETSc








if not os.path.exists("data_poisson"):
    os.makedirs("data_poisson")

N=1000
r1=25e-3+(15e-3)*np.random.rand(N)
r2=((25e-3)**2)/r1

angle=angle=np.zeros(N)
for n in range(N):

    
    
    domain, ft=draw_mesh(r1[n],r2[n], angle[n])

    
    
    
    
   

    V = fem.functionspace(domain, ("Lagrange", 1))

    tdim = domain.topology.dim
    fdim = tdim - 1
    

    
    bc_zeros = fem.dirichletbc(PETSc.ScalarType(0), fem.locate_dofs_topological(V, fdim, ft.find(1)), V)
    
    bc_non_zero1 = fem.dirichletbc(PETSc.ScalarType(-1), fem.locate_dofs_topological(V, fdim, ft.find(2)), V)
    bc_non_zero2 = fem.dirichletbc(PETSc.ScalarType(1), fem.locate_dofs_topological(V, fdim, ft.find(3)), V)

    bc = [bc_zeros, bc_non_zero1, bc_non_zero2]

    u = ufl.TrialFunction(V)
    v = ufl.TestFunction(V)
    f = fem.Constant(domain, PETSc.ScalarType(0))

    a = ufl.dot(ufl.grad(u), ufl.grad(v)) * ufl.dx
    L = f * v * ufl.dx


    problem = LinearProblem(a, L, bcs=bc, petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
    uh = problem.solve()

    e_field=ufl.grad(uh)
    abs_e=ufl.sqrt(ufl.inner(e_field,e_field))
    V_abs_e=fem.functionspace(domain, ("DG", 0))
    e_expr = fem.Expression(abs_e, V_abs_e.element.interpolation_points())
    e = fem.Function(V_abs_e)
    e.interpolate(e_expr)
    
    

    cell_to_vertices = domain.topology.connectivity(domain.topology.dim, 0).array.reshape(-1,3)
    points= domain.geometry.x.reshape(-1,3)

    bb_tree = geometry.bb_tree(domain, domain.topology.dim)
    cells = []
    points_on_proc = []
    # Find cells whose bounding-box collide with the the points
    cell_candidates = geometry.compute_collisions_points(bb_tree, points)
    # Choose one of the cells that contains the point
    colliding_cells = geometry.compute_colliding_cells(domain, cell_candidates, points)
    for i, point in enumerate(points):
        if len(colliding_cells.links(i)) > 0:
            points_on_proc.append(point)
            cells.append(colliding_cells.links(i)[0])
    val=e.eval(domain.geometry.x, cells)

    

        
    if not os.path.exists("data_poisson/"+str(n)):
        os.makedirs("data_poisson/"+str(n))
    np.save("data_poisson/"+str(n)+"/H_holes.npy", np.array([[80e-3,150e-3, r1[n],r2[n],angle[n]],[170e-3,150e-3,r1[n],r2[n],angle[n]],[80e-3, 250e-3, r1[n],r2[n],angle[n]],[170e-3, 250e-3, r1[n],r2[n],angle[n]],[80e-3, 350e-3, r1[n],r2[n],angle[n]],[170e-3, 350e-3, r1[n],r2[n],angle[n]]]))
    
    np.save("data_poisson/"+str(n)+"/H_mesh_geometry.npy", points[:,:2])
    np.save("data_poisson/"+str(n)+"/H_mesh_topology.npy", cell_to_vertices)
    np.save("data_poisson/"+str(n)+"/H_y.npy", val)

    
   


        

    
    


Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 20%] Meshing curve 2 (Line)
Info    : [ 30%] Meshing curve 3 (Line)
Info    : [ 40%] Meshing curve 4 (Line)
Info    : [ 50%] Meshing curve 5 (Ellipse)
Info    : [ 60%] Meshing curve 6 (Ellipse)
Info    : [ 70%] Meshing curve 7 (Ellipse)
Info    : [ 80%] Meshing curve 8 (Ellipse)
Info    : [ 90%] Meshing curve 9 (Ellipse)
Info    : [100%] Meshing curve 10 (Ellipse)
Info    : Done meshing 1D (Wall 0.0200165s, CPU 0.020394s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.133654s, CPU 0.132846s)
Info    : 4030 nodes 8074 elements
Info    : Optimizing mesh (Netgen)...
Info    : Done optimizing mesh (Wall 1.1e-06s, CPU 3e-06s)
Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    : Meshing 1D...
Info    : [  0%] Meshing curve

In [5]:
#x,y=np.meshgrid(np.array([0,2,4]),np.array([1,3,5,7,9]))
#x.reshape(-1).reshape(5,3)

In [6]:
#import matplotlib.pyplot as plt
#plt.contourf(x, y, final_val.reshape(500,250), 800)

In [7]:
def draw_mesh(r1, r2,angle):
    gmsh.initialize()
    gmsh.clear()
    gmsh.model.add("test")
    gmsh.model.setCurrent("test")

    gmsh.model.occ.addPoint(250e-3,0,0,meshSize=40e-3, tag=1)
    gmsh.model.occ.addPoint(250e-3, 500e-3, 0, meshSize=40e-3, tag=2)
    gmsh.model.occ.addPoint(0,0,0,meshSize=40e-3, tag=3)
    gmsh.model.occ.addPoint(0,500e-3,0, meshSize=40e-3, tag=4)

    gmsh.model.occ.addLine(3,1,tag=1)
    gmsh.model.occ.addLine(3,4,tag=2)
    gmsh.model.occ.addLine(1,2,tag=3)
    gmsh.model.occ.addLine(2,4,tag=4)

    gmsh.model.occ.addCurveLoop([3,4,-2,1], tag=1)


    
    gmsh.model.occ.addPoint(80e-3+r1, 150e-3, 0, meshSize=16e-3, tag=20)   
    gmsh.model.occ.addEllipse(80e-3,150e-3,0, r1,r2, tag=5, angle1=0, angle2=2*np.pi,) 
    gmsh.model.occ.rotate([(0,20),(1,5)],80e-3,150e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([5], tag=2)

    gmsh.model.occ.addPoint(170e-3+r1, 150e-3, 0, meshSize=16e-3, tag=22)
    gmsh.model.occ.addEllipse(170e-3,150e-3,0, r1,r2, tag=6, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,22),(1,6)],170e-3,150e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([6], tag=3)

    gmsh.model.occ.addPoint(80e-3+r1, 250e-3, 0, meshSize=16e-3, tag=24)
    gmsh.model.occ.addEllipse(80e-3,250e-3,0, r1,r2, tag=7, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,24),(1,7)],80e-3,250e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([7], tag=4)

    
    gmsh.model.occ.addPoint(170e-3+r1, 250e-3, 0, meshSize=16e-3, tag=26)
    gmsh.model.occ.addEllipse(170e-3,250e-3,0, r1,r2, tag=8, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,26),(1,8)],170e-3,250e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([8], tag=5)

    gmsh.model.occ.addPoint(80e-3+r1, 350e-3, 0, meshSize=16e-3, tag=28)
    gmsh.model.occ.addEllipse(80e-3,350e-3,0, r1,r2, tag=9, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,28),(1,9)],80e-3,350e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([9], tag=6)

    gmsh.model.occ.addPoint(170e-3+r1, 350e-3, 0, meshSize=16e-3, tag=30)
    gmsh.model.occ.addEllipse(170e-3,350e-3,0, r1,r2, tag=10, angle1=0, angle2=2*np.pi,)
    gmsh.model.occ.rotate([(0,30),(1,10)],170e-3,350e-3,0,0,0,1,angle/180*np.pi)
    gmsh.model.occ.addCurveLoop([10], tag=7)


    gmsh.model.occ.addPlaneSurface([1,2,3,4,5,6,7], tag=1)


    gmsh.model.occ.synchronize()

    gmsh.model.addPhysicalGroup(1, [1,2,3,4], 1, "V=0")
    gmsh.model.addPhysicalGroup(1, [5,8,9], 2, "V=-1")
    gmsh.model.addPhysicalGroup(1, [6,7,10], 3, "V=1")
    
    
    
    gmsh.model.addPhysicalGroup(2, [1], 5, "domain")

    gmsh.model.mesh.generate(2)
    gmsh.model.mesh.optimize("Netgen")

    model_rank = 0
    mesh_comm = MPI.COMM_WORLD
    domain, ct, ft = gmshio.model_to_mesh(gmsh.model, mesh_comm, model_rank, gdim=2)
    gmsh.finalize()
    return domain, ft
    
    

In [8]:
"""
pyvista.start_xvfb()
plotter = pyvista.Plotter()
tdim = domain.topology.dim
domain.topology.create_connectivity(tdim, tdim)
grid = pyvista.UnstructuredGrid(*vtk_mesh(domain, tdim))
num_local_cells = domain.topology.index_map(tdim).size_local
grid.cell_data["Marker"] = ct.values[ct.indices < num_local_cells]
grid.set_active_scalars("Marker")
actor = plotter.add_mesh(grid, show_edges=True, color="white")
plotter.view_xy()

plotter.show()
"""

'\npyvista.start_xvfb()\nplotter = pyvista.Plotter()\ntdim = domain.topology.dim\ndomain.topology.create_connectivity(tdim, tdim)\ngrid = pyvista.UnstructuredGrid(*vtk_mesh(domain, tdim))\nnum_local_cells = domain.topology.index_map(tdim).size_local\ngrid.cell_data["Marker"] = ct.values[ct.indices < num_local_cells]\ngrid.set_active_scalars("Marker")\nactor = plotter.add_mesh(grid, show_edges=True, color="white")\nplotter.view_xy()\n\nplotter.show()\n'

In [9]:
if not os.path.exists("data_poisson"):
    os.makedirs("data_poisson")


for n in range(N):

    
    
    domain, ft=draw_mesh(r1[n],r2[n], angle[n])

    V = fem.functionspace(domain, ("Lagrange", 1))

    tdim = domain.topology.dim
    fdim = tdim - 1
    
    bc_zeros = fem.dirichletbc(PETSc.ScalarType(0), fem.locate_dofs_topological(V, fdim, ft.find(1)), V)
    
    bc_non_zero1 = fem.dirichletbc(PETSc.ScalarType(-1), fem.locate_dofs_topological(V, fdim, ft.find(3)), V)
    bc_non_zero2 = fem.dirichletbc(PETSc.ScalarType(1), fem.locate_dofs_topological(V, fdim, ft.find(2)), V)
  
    bc = [bc_zeros, bc_non_zero1, bc_non_zero2]

    u = ufl.TrialFunction(V)
    v = ufl.TestFunction(V)
    f = fem.Constant(domain, PETSc.ScalarType(0))

    a = ufl.dot(ufl.grad(u), ufl.grad(v)) * ufl.dx
    L = f * v * ufl.dx


    problem = LinearProblem(a, L, bcs=bc, petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
    uh = problem.solve()

    e_field=ufl.grad(uh)
    abs_e=ufl.sqrt(ufl.inner(e_field,e_field))
    V_abs_e=fem.functionspace(domain, ("DG", 0))
    e_expr = fem.Expression(abs_e, V_abs_e.element.interpolation_points())
    e = fem.Function(V_abs_e)
    e.interpolate(e_expr)
    
    

    cell_to_vertices = domain.topology.connectivity(domain.topology.dim, 0).array.reshape(-1,3)
    points= domain.geometry.x.reshape(-1,3)

    bb_tree = geometry.bb_tree(domain, domain.topology.dim)
    cells = []
    points_on_proc = []
    # Find cells whose bounding-box collide with the the points
    cell_candidates = geometry.compute_collisions_points(bb_tree, points)
    # Choose one of the cells that contains the point
    colliding_cells = geometry.compute_colliding_cells(domain, cell_candidates, points)
    for i, point in enumerate(points):
        if len(colliding_cells.links(i)) > 0:
            points_on_proc.append(point)
            cells.append(colliding_cells.links(i)[0])
    val=e.eval(domain.geometry.x, cells)

        
    if not os.path.exists("data_poisson/"+str(n)):
        os.makedirs("data_poisson/"+str(n))
    np.save("data_poisson/"+str(n)+"/L_holes.npy", np.array([[80e-3,150e-3, r1[n],r2[n],angle[n]],[170e-3,150e-3,r1[n],r2[n],angle[n]],[80e-3, 250e-3, r1[n],r2[n],angle[n]],[170e-3, 250e-3, r1[n],r2[n],angle[n]],[80e-3, 350e-3, r1[n],r2[n],angle[n]],[170e-3, 350e-3, r1[n],r2[n],angle[n]]]))
    
    np.save("data_poisson/"+str(n)+"/L_mesh_geometry.npy", points[:,:2])
    np.save("data_poisson/"+str(n)+"/L_mesh_topology.npy", cell_to_vertices)
    np.save("data_poisson/"+str(n)+"/L_y.npy", val)

Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 20%] Meshing curve 2 (Line)
Info    : [ 30%] Meshing curve 3 (Line)
Info    : [ 40%] Meshing curve 4 (Line)
Info    : [ 50%] Meshing curve 5 (Ellipse)
Info    : [ 60%] Meshing curve 6 (Ellipse)
Info    : [ 70%] Meshing curve 7 (Ellipse)
Info    : [ 80%] Meshing curve 8 (Ellipse)
Info    : [ 90%] Meshing curve 9 (Ellipse)
Info    : [100%] Meshing curve 10 (Ellipse)
Info    : Done meshing 1D (Wall 0.0165073s, CPU 0.016974s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.024227s, CPU 0.024608s)
Info    : 352 nodes 718 elements
Info    : Optimizing mesh (Netgen)...
Info    : Done optimizing mesh (Wall 1.1e-06s, CPU 1e-06s)
Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1

In [10]:
"""
pyvista.start_xvfb()

# Create plotter and pyvista grid
p = pyvista.Plotter()
topology, cell_types, geo = plot.vtk_mesh(V)
grid = pyvista.UnstructuredGrid(topology, cell_types, geo)

# Attach vector values to grid and warp grid by vector
grid["VonMises"] = stresses.x.array
grid.set_active_scalars("VonMises")
p.add_mesh(grid, show_edges=False)

p.show_axes()

p.show()
"""

'\npyvista.start_xvfb()\n\n# Create plotter and pyvista grid\np = pyvista.Plotter()\ntopology, cell_types, geo = plot.vtk_mesh(V)\ngrid = pyvista.UnstructuredGrid(topology, cell_types, geo)\n\n# Attach vector values to grid and warp grid by vector\ngrid["VonMises"] = stresses.x.array\ngrid.set_active_scalars("VonMises")\np.add_mesh(grid, show_edges=False)\n\np.show_axes()\n\np.show()\n'